# CAPTURe Real x Moondream2 -- capture_fix (prompt oficial amodal)

**Modelo:** Moondream2 (1.8B) — `vikhyatk/moondream2`  
**Benchmark:** CAPTURe **real** — 824 imágenes, objetos cotidianos, conteo 8–907  
**Salida:** `tabla_completa.csv` (una fila por imagen) + `extras.csv` (resumen de la corrida)  

---
### Cambios respecto al piloto anterior
| Qué cambió | Piloto | Esta versión |
|---|---|---|
| Dataset | n=50 (muestra) | n=824 (completo) |
| Subset | real + sintético juntos | solo real |
| Salida | `.txt` + `detalle.csv` | `tabla_completa.csv` + `extras.csv` |
| Columnas nuevas | — | `image_file`, `bucket`, `error_relativo`, `modelo` |

El resto del pipeline (parseo, timeout, `predecir()`) es **idéntico** al piloto.

---
**Fix aplicado sobre la version anterior:** se reemplazo el prompt "How many {object} are there?" + sufijo "Respond with only the integer number" (eso media conteo denso, no conteo amodal) por el prompt oficial del paper CAPTURe (pide contar asumiendo que el patron continua detras del recuadro negro, que ya viene quemado en la imagen). No se toco descarga, muestreo, parseo ni guardado -- esas partes ya eran correctas, incluyendo `image_file` en `tabla_completa.csv`, que ya se guardaba en esta version del notebook.


## 1. Instalación
Misma versión que el piloto (`transformers==4.57.6`). Reiniciar sesión si la celda 2 da error de versión.

In [ ]:
!pip install -q "transformers==4.57.6" accelerate einops timm pillow psutil huggingface_hub
print('Dependencias instaladas.')

In [ ]:
import transformers
print(f'transformers: {transformers.__version__}')
if not transformers.__version__.startswith('4.'):
    raise RuntimeError('Versión incompatible. Entorno de ejecución → Reiniciar sesión → Ejecutar todo.')
print('Versión compatible. Continuando...')

In [ ]:
# ── Montar Google Drive ────────────────────────────────────────────────────
# Esto guarda los CSV directamente en tu Drive, sin necesidad de descargar
# manualmente. Si el notebook está en Drive, los archivos quedan en la misma
# carpeta que el notebook.
from google.colab import drive
drive.mount('/content/drive')

import os

# Ruta base dentro de Drive — ajustar si tt2 no está en la raíz de 'My Drive'
# Ejemplo si está en My Drive/tt2/capture/real/moondream2/:
DRIVE_NOTEBOOK_DIR = '/content/drive/MyDrive/tt2/capture/real/moondream2'

os.makedirs(DRIVE_NOTEBOOK_DIR, exist_ok=True)
print(f'Drive montado. Carpeta de salida: {DRIVE_NOTEBOOK_DIR}')
print('Archivos existentes en esa carpeta:', os.listdir(DRIVE_NOTEBOOK_DIR))

## 2. Configuración
**Solo cambiar `COLAB_TIPO` si no es Pro.** El resto no tocar para mantener reproducibilidad.

In [ ]:
MODEL_ID       = 'vikhyatk/moondream2'
MODEL_REVISION = '2025-06-21'
MODEL_LABEL    = 'moondream2'          # identificador corto para los CSV

COLAB_TIPO     = 'Pro'                 # 'Free' | 'Pro' | 'Pro+' — completar manualmente
RANDOM_SEED    = 42
TIMEOUT_SEG    = 30
PROMPT_SUFFIX  = ''  # oficial: no se agrega sufijo, el prompt ya pide el conteo directamente

# Los CSV se guardan directamente en Drive (ruta definida en la celda anterior)
OUTPUT_TABLA   = f'{DRIVE_NOTEBOOK_DIR}/tabla_completa.csv'
OUTPUT_EXTRAS  = f'{DRIVE_NOTEBOOK_DIR}/extras.csv'

# Buckets de magnitud para CAPTURe real (basados en cuartiles del dataset completo)
BUCKETS = [('8-15', 8, 15), ('16-28', 16, 28), ('29-67', 29, 67), ('68-907', 68, 907)]

def asignar_bucket(gt):
    for nombre, lo, hi in BUCKETS:
        if lo <= gt <= hi: return nombre
    return 'fuera_de_rango'

print(f'Modelo:  {MODEL_ID} (revision={MODEL_REVISION})')
print(f'Colab:   {COLAB_TIPO}')
print(f'Seed:    {RANDOM_SEED}  |  Timeout: {TIMEOUT_SEG}s')

## 3. Cargar modelo
**Este bloque es el único que cambia entre modelos.** Todo lo demás del notebook es idéntico para Qwen2-VL, InternVL2 y PaliGemma2.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')
if device == 'cpu':
    print('ADVERTENCIA: sin GPU. Activar en Entorno de ejecución → Cambiar tipo de entorno.')

if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, trust_remote_code=True, revision=MODEL_REVISION,
    dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
    device_map={'': device},
)
print(f'Modelo cargado. VRAM tras carga: {torch.cuda.max_memory_allocated()/1024**2:.1f} MB')

# ── Interfaz común (único punto que difiere entre modelos) ──────
def predecir(imagen_pil, pregunta_texto):
    """Recibe PIL.Image y str → devuelve str (respuesta cruda)."""
    return model.query(imagen_pil, pregunta_texto)['answer']

# Diagnóstico
print('torch.cuda.is_available():', torch.cuda.is_available())
print('Dispositivo del modelo:', next(model.parameters()).device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader 2>/dev/null || echo 'nvidia-smi no disponible'

## 4. Descargar CAPTURe real
Solo se descarga el subset **real** (no el sintético). Los zips están en HuggingFace Hub.

In [ ]:
import os, zipfile, pandas as pd
from huggingface_hub import hf_hub_download

print('Descargando real_dataset.zip...')
real_zip  = hf_hub_download(repo_id='atinp/CAPTURe', filename='real_dataset.zip',   repo_type='dataset')
print('Descargando real_metadata.json...')
real_meta = hf_hub_download(repo_id='atinp/CAPTURe', filename='real_metadata.json', repo_type='dataset')

REAL_IMAGE_DIR = '/content/real_dataset/'
with zipfile.ZipFile(real_zip, 'r') as z:
    z.extractall()
if os.path.exists('dataset') and not os.path.exists('real_dataset'):
    os.rename('dataset', 'real_dataset')

real_df = pd.read_json(real_meta)
print(f'Metadata cargada: {len(real_df)} imágenes | columnas: {list(real_df.columns)}')
real_df.head(3)

## 5. Construir items
Dataset completo (n=824). Se agrega `image_file` y se precalcula `bucket` para no tener que re-derivarlos después.

In [ ]:
def construir_item(row):
    return {
        'image_file':   row['image_file'],
        'image_path':   os.path.join(REAL_IMAGE_DIR, row['image_file']),
        'pregunta':     (f"Count the exact number of {row['object']} in the image. "
                    f"Assume the pattern of {row['object']} continues behind any black box. "
                    f"Provide the total number of {row['object']} as if the black box were not there."),
        'categoria':    row['object'],
        'ground_truth': int(row['ground_truth']),
        'bucket':       asignar_bucket(int(row['ground_truth'])),
    }

muestra = [construir_item(row) for _, row in real_df.iterrows()]
print(f'Items construidos: {len(muestra)}')
print(f'Distribución por bucket:')
from collections import Counter
for b, n in sorted(Counter(i['bucket'] for i in muestra).items()):
    print(f'  {b}: {n}')

## 6. Prueba visual rápida
Verificar que las imágenes cargan y el modelo responde antes del loop completo.

In [ ]:
from IPython.display import display
import time

item_prueba = muestra[0]
img_prueba  = Image.open(item_prueba['image_path']).convert('RGB')
t0 = time.time()
resp_prueba = predecir(img_prueba, item_prueba['pregunta'])  # prompt oficial ya completo, sin sufijo
print(f"Pregunta: {item_prueba['pregunta']}")
print(f"GT: {item_prueba['ground_truth']} | Respuesta: '{resp_prueba}' | t={time.time()-t0:.2f}s")
display(img_prueba)

## 7. Loop de inferencia
Idéntico al piloto. Parser: dígito primero, palabras como fallback. Timeout por ThreadPoolExecutor.

In [ ]:
import re, time, concurrent.futures

NUMEROS_EN_PALABRAS = {
    'zero': 0, 'none': 0, 'no': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
    'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10,
    'eleven': 11, 'twelve': 12, 'thirteen': 13, 'fourteen': 14, 'fifteen': 15,
    'sixteen': 16, 'seventeen': 17, 'eighteen': 18, 'nineteen': 19, 'twenty': 20,
}

def extraer_entero(texto):
    if texto is None: return None
    m = re.search(r'-?\d+', texto)
    if m: return int(m.group())
    limpio = re.sub(r'[^\w\s]', ' ', texto.lower())
    for palabra, valor in NUMEROS_EN_PALABRAS.items():
        if re.search(rf'\b{palabra}\b', limpio): return valor
    return None

def query_con_timeout(imagen, pregunta, timeout_seg):
    ex = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(predecir, imagen, pregunta)
    try:
        res = fut.result(timeout=timeout_seg)
        ex.shutdown(wait=False)
        return res, False
    except concurrent.futures.TimeoutError:
        ex.shutdown(wait=False)
        return None, True

resultados = []
n_timeouts = 0
print(f'Iniciando loop sobre {len(muestra)} imágenes (timeout={TIMEOUT_SEG}s)...\n')

for i, item in enumerate(muestra):
    try:
        img = Image.open(item['image_path']).convert('RGB')
    except Exception as e:
        print(f'[{i+1}] ERROR cargando imagen {item["image_file"]}: {e}')
        continue

    t0 = time.time()
    respuesta_cruda, timed_out = query_con_timeout(img, item['pregunta'], TIMEOUT_SEG)  # prompt oficial ya completo
    t1 = time.time()

    if timed_out:
        n_timeouts += 1
        pred, parse_fail = None, False
    else:
        pred = extraer_entero(respuesta_cruda)
        parse_fail = (pred is None)

    gt  = item['ground_truth']
    err = abs(pred - gt) if pred is not None else None

    # ── Dict con TODAS las columnas de tabla_completa ─────────
    resultados.append({
        'modelo':          MODEL_LABEL,
        'image_file':      item['image_file'],
        'subset':          'real',
        'categoria':       item['categoria'],
        'bucket':          item['bucket'],
        'ground_truth':    gt,
        'respuesta_cruda': respuesta_cruda,
        'prediccion':      pred,
        'parse_fail':      parse_fail,
        'timeout':         timed_out,
        'acierto_exacto':  (pred == gt) if pred is not None else False,
        'error_abs':       err,
        'error_relativo':  round(err / gt, 4) if (err is not None and gt > 0) else None,
        'tiempo_seg':      round(t1 - t0, 4),
    })

    if timed_out:
        print(f'[{i+1:4d}/{len(muestra)}] TIMEOUT {item["image_file"]}')
    elif (i + 1) % 50 == 0 or i == 0:
        print(f'[{i+1:4d}/{len(muestra)}] {item["categoria"]:25s} GT={gt:4d} pred={str(pred):>5} ({t1-t0:.2f}s)')

print(f'\nLoop completo. {len(resultados)} procesadas, {n_timeouts} timeouts.')

## 8. Métricas rápidas
Resumen en pantalla. Los números definitivos vienen del script de análisis sobre `tabla_completa.csv`.

In [ ]:
import pandas as pd, psutil

df = pd.DataFrame(resultados)
validos = df[~df['parse_fail'] & ~df['timeout']]

vram_mb = torch.cuda.max_memory_allocated() / 1024**2 if device == 'cuda' else 0
ram_mb  = psutil.Process().memory_info().rss / 1024**2

print(f'N total:          {len(df)}')
print(f'Timeout rate:     {df["timeout"].mean():.2%}')
print(f'Parse fail rate:  {df["parse_fail"].mean():.2%}')
print(f'Accuracy exacta:  {validos["acierto_exacto"].mean():.2%}')
print(f'MAE:              {validos["error_abs"].mean():.3f}')
print(f'Tiempo total:     {df["tiempo_seg"].sum():.1f}s ({df["tiempo_seg"].sum()/60:.1f} min)')
print(f'Tiempo medio/img: {df["tiempo_seg"].mean():.3f}s')
print(f'VRAM pico:        {vram_mb:.1f} MB')
print(f'RAM pico:         {ram_mb:.1f} MB')
print()
print('Desglose por bucket:')
for b, _, _ in BUCKETS:
    sub = validos[validos['bucket'] == b]
    if len(sub): print(f'  {b:8s}: n={len(sub):4d}  acc={sub["acierto_exacto"].mean():.1%}  MAE={sub["error_abs"].mean():.2f}')

## 9. Guardar `tabla_completa.csv`
Una fila por imagen. Orden de columnas fijo para consistencia entre modelos.

In [ ]:
COLS_TABLA = [
    'modelo', 'image_file', 'subset', 'categoria', 'bucket',
    'ground_truth', 'respuesta_cruda', 'prediccion',
    'parse_fail', 'timeout', 'acierto_exacto',
    'error_abs', 'error_relativo', 'tiempo_seg',
]
df[COLS_TABLA].to_csv(OUTPUT_TABLA, index=False)
print(f'tabla_completa.csv guardado: {len(df)} filas × {len(COLS_TABLA)} columnas')
print(df[COLS_TABLA].head(3).to_string())

## 10. Guardar `extras.csv`
Una sola fila con los metadatos de la corrida completa: hardware, VRAM, RAM, tiempos totales y métricas globales.

In [ ]:
import datetime, pandas as pd

try:
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
except Exception:
    gpu_name = 'desconocido'

validos = df[~df['parse_fail'] & ~df['timeout']]

extras = pd.DataFrame([{
    'modelo':              MODEL_LABEL,
    'benchmark':           'capture_real',
    'fecha':               datetime.datetime.now().isoformat(),
    'hardware_gpu':        gpu_name,
    'colab_tipo':          COLAB_TIPO,
    'n_muestras':          len(df),
    'random_seed':         RANDOM_SEED,
    'tiempo_total_seg':    round(df['tiempo_seg'].sum(), 2),
    'tiempo_medio_seg':    round(df['tiempo_seg'].mean(), 4),
    'vram_pico_mb':        round(vram_mb, 2),
    'ram_pico_mb':         round(ram_mb, 2),
    'timeout_rate':        round(df['timeout'].mean(), 6),
    'parse_fail_rate':     round(df['parse_fail'].mean(), 6),
    'accuracy_exacta':     round(validos['acierto_exacto'].mean(), 6),
    'mae':                 round(validos['error_abs'].mean(), 4),
    'transformers_version': transformers.__version__,
    'notas':               '',
}])

extras.to_csv(OUTPUT_EXTRAS, index=False)
print('extras.csv guardado:')
print(extras.T.to_string())

## 11. Descargar archivos

In [ ]:
# Los archivos ya quedaron guardados en Drive automáticamente.
# Si además querés una copia local (descarga al navegador), descomenta las líneas:
# from google.colab import files
# files.download(OUTPUT_TABLA)
# files.download(OUTPUT_EXTRAS)
print(f'✓ tabla_completa.csv → {OUTPUT_TABLA}')
print(f'✓ extras.csv         → {OUTPUT_EXTRAS}')
print()
print('Archivos en la carpeta de Drive:')
import os
for f in sorted(os.listdir(DRIVE_NOTEBOOK_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_NOTEBOOK_DIR, f))
    print(f'  {f}  ({size/1024:.1f} KB)')